# Cross-Dataset Benchmark — Run 2 (YOLOv11n / Kaggle) → DeepPCB

**Goal:** measure how well the Run 2 model (trained on the Kaggle `norbertelter/pcb-defect-dataset`)
generalizes to a *different* PCB defect dataset it has never seen: **DeepPCB**.

DeepPCB shares the same 6 trace-defect classes but is a fully independent source with
binarized template-difference imaging — a real domain shift. The gap between Run 2's
in-domain mAP and its DeepPCB mAP is the first datapoint of the generalization benchmark.

This notebook only **evaluates** an existing model — no training. Run cells top to bottom.

## 1. Environment check

In [ ]:
import torch
print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))

In [ ]:
import subprocess, sys
try:
    import ultralytics
except ImportError:
    subprocess.run([sys.executable,"-m","pip","install","ultralytics","-q"], check=True)
    import ultralytics
import yaml as _yaml  # pyyaml ships with ultralytics
print("ultralytics:", ultralytics.__version__)

## 2. Configuration

Edit `RUN2_WEIGHTS` if your trained `best.pt` lives elsewhere (it is git-ignored, so it must come from your local Run 2 training output).

In [ ]:
from pathlib import Path

import os
ROOT_ENV = os.environ.get("PCB_PROJECT_ROOT")  # optional override
_cwd = Path(os.getcwd()).resolve()
if ROOT_ENV:
    PROJECT_ROOT = Path(ROOT_ENV).resolve()
elif _cwd.name == "experiments":
    PROJECT_ROOT = _cwd.parent
elif (_cwd / "experiments").exists():
    PROJECT_ROOT = _cwd
else:
    PROJECT_ROOT = _cwd  # fallback: assume cwd is project root
RUN2_WEIGHTS = PROJECT_ROOT / "results" / "exp_002_yolov11n_kaggle_pcb_200ep" / "weights" / "best.pt"
SEED, IMG_SIZE, BATCH, DEVICE = 42, 640, 16, 0     # DEVICE='cpu' if no GPU

# Run 2 model class order (index -> name) — this is how Run 2 was TRAINED.
RUN2_NAMES = {0:'mouse_bite', 1:'spur', 2:'missing_hole', 3:'short', 4:'open_circuit', 5:'spurious_copper'}

# DeepPCB annotation type (1-6) -> Run 2 model class index.
# DeepPCB key: 1=open, 2=short, 3=mousebite, 4=spur, 5=copper(=spurious_copper), 6=pin-hole(=missing_hole)
DEEPPCB_TO_RUN2 = {1:4, 2:3, 3:0, 4:1, 5:5, 6:2}

DEEPPCB_DIR = PROJECT_ROOT / "DeepPCB"
OUT_DIR     = PROJECT_ROOT / "deeppcb_yolo"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root      :", PROJECT_ROOT)
print("Run2 weights found:", RUN2_WEIGHTS.exists(), "->", RUN2_WEIGHTS)

## 3. Download DeepPCB

~a few hundred MB. If `git` isn't installed, download the ZIP from https://github.com/tangsanli5201/DeepPCB and unzip into the `DeepPCB` folder.

In [ ]:
import subprocess
if not DEEPPCB_DIR.exists():
    print("Cloning DeepPCB ...")
    r = subprocess.run(["git","clone","--depth","1","https://github.com/tangsanli5201/DeepPCB", str(DEEPPCB_DIR)],
                       capture_output=True, text=True)
    print((r.stdout or "")[-400:]); print((r.stderr or "")[-400:])
    if r.returncode != 0:
        raise RuntimeError("git clone failed. Install git OR download the repo ZIP manually and unzip to "+str(DEEPPCB_DIR))
else:
    print("DeepPCB already present.")

PCBDATA = DEEPPCB_DIR / "PCBData"
assert PCBDATA.exists(), f"Expected {PCBDATA}. Top-level contents: {[p.name for p in DEEPPCB_DIR.iterdir()]}"
print("PCBData OK. Top-level:", [p.name for p in DEEPPCB_DIR.iterdir()])

## 4. Locate the official test split list

In [ ]:
list_files = list(DEEPPCB_DIR.rglob("test.txt")) + list(DEEPPCB_DIR.rglob("trainval.txt"))
print("Found list files:", [str(p.relative_to(DEEPPCB_DIR)) for p in list_files])

test_list = next((p for p in list_files if p.name == "test.txt"), None)
assert test_list is not None, "test.txt not found in DeepPCB."
print("\nUsing:", test_list.relative_to(DEEPPCB_DIR))
print("First lines (to confirm format = '<img> <annotation>'):")
for ln in test_list.read_text().splitlines()[:3]:
    print("  ", repr(ln))

## 5. Convert DeepPCB test set → YOLO format, in the Run 2 label space

Boxes are remapped to your model's class indices so evaluation is meaningful.

In [ ]:
from PIL import Image
import shutil

def resolve(rel):           # list paths are relative to PCBData
    return PCBDATA / rel

def parse_line(line):
    p = line.strip().split()
    if not p: return None
    img_rel = p[0]
    ann_rel = p[1] if len(p) > 1 else img_rel.replace("_test.jpg", ".txt")
    return img_rel, ann_rel

img_out = OUT_DIR / "test" / "images"; img_out.mkdir(parents=True, exist_ok=True)
lbl_out = OUT_DIR / "test" / "labels"; lbl_out.mkdir(parents=True, exist_ok=True)

n_img=n_box=skipped=0; unknown=set()
for line in test_list.read_text().splitlines():
    parsed = parse_line(line)
    if not parsed: continue
    img_rel, ann_rel = parsed
    img_p, ann_p = resolve(img_rel), resolve(ann_rel)
    if not img_p.exists():                                  # fallback: glob by filename
        cand = list(PCBDATA.rglob(Path(img_rel).name)); img_p = cand[0] if cand else img_p
    if not ann_p.exists():
        cand = list(PCBDATA.rglob(Path(ann_rel).name)); ann_p = cand[0] if cand else ann_p
    if not img_p.exists() or not ann_p.exists():
        skipped += 1; continue
    with Image.open(img_p) as im: W,H = im.size
    rows=[]
    for raw in ann_p.read_text().splitlines():
        v = raw.split()
        if len(v) < 5: continue
        x1,y1,x2,y2,t = float(v[0]),float(v[1]),float(v[2]),float(v[3]),int(float(v[4]))
        if t not in DEEPPCB_TO_RUN2: unknown.add(t); continue
        cx,cy,bw,bh = (x1+x2)/2/W,(y1+y2)/2/H,(x2-x1)/W,(y2-y1)/H
        if bw<=0 or bh<=0: continue
        rows.append(f"{DEEPPCB_TO_RUN2[t]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}"); n_box+=1
    stem = img_p.stem
    shutil.copy2(img_p, img_out / f"{stem}.jpg")
    (lbl_out / f"{stem}.txt").write_text("\n".join(rows))
    n_img += 1

print(f"Converted {n_img} images, {n_box} boxes. Skipped(missing): {skipped}")
if unknown: print("WARNING: unmapped DeepPCB types:", unknown)
assert n_img>0 and n_box>0, "Conversion produced nothing — inspect the list-file format above."

## 6. Write the DeepPCB `data.yaml` (names in Run 2 order)

In [ ]:
import yaml
deeppcb_yaml = OUT_DIR / "deeppcb.yaml"
deeppcb_yaml.write_text(yaml.dump({
    "path": str(OUT_DIR),
    "train":"test/images", "val":"test/images", "test":"test/images",
    "names": {int(k):v for k,v in RUN2_NAMES.items()},
}, sort_keys=False))
print(deeppcb_yaml.read_text())

## 7. Load Run 2 model and VERIFY class alignment

If this warns about a mismatch, the numbers below are invalid — fix `RUN2_NAMES`/`DEEPPCB_TO_RUN2` first.

In [ ]:
from ultralytics import YOLO
assert RUN2_WEIGHTS.exists(), f"Run2 best.pt not found at {RUN2_WEIGHTS}. It's git-ignored; point to your local trained weights."
model = YOLO(str(RUN2_WEIGHTS))
print("Model class names:", model.names)
mismatch = {i:(model.names.get(i), RUN2_NAMES.get(i)) for i in range(6) if model.names.get(i)!=RUN2_NAMES.get(i)}
print("\nClass-order check:", "MISMATCH -> FIX BEFORE TRUSTING METRICS: "+str(mismatch) if mismatch else "verified, model matches DeepPCB label space.")

## 8. Cross-domain evaluation: Run 2 → DeepPCB

In [ ]:
import pandas as pd
cross = model.val(data=str(deeppcb_yaml), split="test", imgsz=IMG_SIZE, batch=BATCH, device=DEVICE,
                  project=str(RESULTS_DIR), name="exp_002_run2_on_DEEPPCB_crossdomain",
                  exist_ok=True, plots=True)

P,R = float(cross.box.mp), float(cross.box.mr)
m50, m = float(cross.box.map50), float(cross.box.map)
print(f"\nRun2 -> DeepPCB (CROSS-DOMAIN):  P={P:.4f}  R={R:.4f}  mAP50={m50:.4f}  mAP50-95={m:.4f}")

idx  = list(cross.box.ap_class_index)
ap50 = {int(c):float(a) for c,a in zip(idx, cross.box.ap50)}
ap   = {int(c):float(a) for c,a in zip(idx, cross.box.ap)}
per_class = pd.DataFrame({
    "Class":   [RUN2_NAMES[i] for i in range(6)],
    "AP@50":   [round(ap50.get(i, float('nan')),4) for i in range(6)],
    "AP@50-95":[round(ap.get(i, float('nan')),4)   for i in range(6)],
})
print("\nPer-class (NaN = class absent / never predicted):")
print(per_class.to_string(index=False))
per_class.to_csv(RESULTS_DIR / "crossdomain_run2_on_deeppcb_perclass.csv", index=False)

## 9. In-domain vs cross-domain — the benchmark table

In [ ]:
indo_csv = RESULTS_DIR / "exp_002_yolov11n_kaggle_pcb_200ep_summary.csv"
if indo_csv.exists():
    s = pd.read_csv(indo_csv); row = s[s["Split"].str.lower()=="test"].iloc[0]
    in_m50, in_m = float(row["mAP@50"]), float(row["mAP@50-95"])
else:
    in_m50, in_m = 0.9931, 0.6348   # from your repo's run2 summary

comp = pd.DataFrame([
    {"Setting":"Run2 in-domain (Kaggle test)",  "mAP@50":round(in_m50,4), "mAP@50-95":round(in_m,4)},
    {"Setting":"Run2 -> DeepPCB (cross-domain)","mAP@50":round(m50,4),    "mAP@50-95":round(m,4)},
])
comp["mAP@50 drop"]    = round(in_m50-m50,4)
comp["mAP@50-95 drop"] = round(in_m-m,4)
out_csv = RESULTS_DIR / "crossdomain_run2_vs_deeppcb.csv"; comp.to_csv(out_csv, index=False)
print(comp.to_string(index=False)); print("\nsaved:", out_csv)

## 10. Qualitative check — GT (green) vs Run 2 predictions (red)

Confirms the conversion is correct and shows the domain gap visually.

In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as mpimg, random
random.seed(SEED)
imgs = random.sample(sorted((OUT_DIR/"test"/"images").glob("*.jpg")), min(6, n_img))
fig, axes = plt.subplots(2,3, figsize=(16,10))
for ax, ip in zip(axes.flat, imgs):
    res = model.predict(str(ip), imgsz=IMG_SIZE, conf=0.25, verbose=False)[0]
    im = mpimg.imread(str(ip)); ax.imshow(im, cmap="gray"); H,W = im.shape[:2]
    lp = OUT_DIR/"test"/"labels"/(ip.stem+".txt")
    if lp.exists():
        for ln in lp.read_text().splitlines():
            if not ln.strip(): continue
            c,cx,cy,bw,bh = ln.split(); cx,cy,bw,bh=map(float,(cx,cy,bw,bh))
            ax.add_patch(plt.Rectangle(((cx-bw/2)*W,(cy-bh/2)*H), bw*W, bh*H, lw=2, edgecolor="lime", facecolor="none"))
    for b in res.boxes:
        x1,y1,x2,y2 = b.xyxy[0].tolist()
        ax.add_patch(plt.Rectangle((x1,y1), x2-x1, y2-y1, lw=1.5, edgecolor="red", facecolor="none"))
        ax.text(x1, max(y1-3,0), f"{RUN2_NAMES[int(b.cls)]} {float(b.conf):.2f}", color="red", fontsize=7)
    ax.set_title(ip.stem, fontsize=8); ax.axis("off")
plt.suptitle("DeepPCB: GT (green) vs Run2 predictions (red) — cross-domain", fontweight="bold")
plt.tight_layout()
qp = RESULTS_DIR / "crossdomain_run2_on_deeppcb_qual.png"; plt.savefig(qp, dpi=120); plt.show()
print("saved:", qp)

## Next steps (build the full benchmark from here)

1. **Reverse direction & third dataset.** Train a model on DeepPCB and test on Kaggle/HRIPCB, and add HRIPCB itself, to fill an N×N train↔test mAP matrix. The off-diagonal collapse is the core result.
2. **Confidence sweep.** Cross-domain optimal confidence differs from in-domain; report mAP plus P/R at a few thresholds.
3. **A domain-generalization method.** Once the gap is quantified, add one intervention (aggressive augmentation, grayscale/edge normalization to match DeepPCB's appearance, or a light adaptation) and show recovery on the off-diagonal.

This notebook is contribution-1 evidence: a controlled, reproducible cross-dataset evaluation of an existing PCB detector.